# 03 - Evaluacion del modelo y explicabilidad

Objetivo: revisar distribucion del score, casos prioritarios, proveedores con concentracion y explicaciones ejecutivas.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from src.ingestion.load_data import load_claims
from src.features.scoring import score_claims
from src.explainability.explain_score import build_executive_explanation

scored = score_claims(load_claims(PROJECT_ROOT / 'data/synthetic/siniestros_sinteticos.csv'))

In [ ]:
summary = {
    'total_siniestros': len(scored),
    'casos_rojos': int((scored['nivel_riesgo'] == 'rojo').sum()),
    'casos_amarillos': int((scored['nivel_riesgo'] == 'amarillo').sum()),
    'casos_verdes': int((scored['nivel_riesgo'] == 'verde').sum()),
    'score_promedio': round(float(scored['score_riesgo'].mean()), 2),
}
summary

In [ ]:
provider_ranking = (
    scored.groupby('beneficiario')
    .agg(
        total_casos=('id_siniestro', 'count'),
        alertas_rojas=('nivel_riesgo', lambda s: int((s == 'rojo').sum())),
        score_promedio=('score_riesgo', 'mean'),
    )
    .sort_values(['alertas_rojas', 'score_promedio'], ascending=False)
    .round(2)
)
provider_ranking

In [ ]:
top_case = scored.sort_values('score_riesgo', ascending=False).iloc[0].to_dict()
build_executive_explanation(top_case)

Criterios de evaluacion cualitativa:

- El modelo debe priorizar revision, no decidir rechazo.
- Las explicaciones deben ser entendibles por analistas.
- Los casos rojos requieren validacion humana y soporte documental.
- Las senales por proveedor o narrativa son indicios, no pruebas concluyentes.